<a target="_blank" href="https://colab.research.google.com/github/AndreiSokolovskii/hackaton_december_2025/blob/main/attention_visual/Attention_Visualisation_notebook.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
%%capture
#@title #Installation required libraries, downloading model weights.
# #@markdown ---
# #@markdown ***only in case of single strtucture prediction***.
# #@markdown ---
# #@markdown
#@markdown This step can take up to ~1 mins
#@markdown


import os, re
if not os.path.isfile("ENV_READY"):
  !pip install torch_geometric
  !pip install pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-2.11.0+cu128.html
  !pip install py3Dmol -q
  !git clone https://github.com/AndreiSokolovskii/hackaton_december_2025.git
  !cp hackaton_december_2025/*.py .
  !tar -xJf hackaton_december_2025/attention_visual/lib.tar.xz
  os.system('touch ENV_READY')

from model_lib import GraphEncoderBlock_fixed
from torch_geometric.data import Batch
import torch_geometric
import os
import copy
from colab_upload_helper import process_upload
import torch
import numpy as np


In [ ]:
#@markdown ##Settings and run.
input = '' #@param {type:"string"}
chains = "A" #@param {type:"string"}
chains = re.sub("[^A-Za-z]+",",", chains)
design_position = 'all'#@param {type:"string"}
path_list = process_upload(pdb_code=input)
file_path = str(path_list)
num_seq_per_target = 1
rm_aa = "" #@param {type:"string"}
#@markdown - `rm_aa='C'` - do not use [C]ysteines.
progress_bar = True #@param {type:"boolean"}
model_noise = 'n03' #@param ["n00", "n01", "n02", "n03", "n05"] {type:"string"}
sampling_temp = "0.1" #@param ["0.01", "0.1", "0.15", "0.2","0.3","0.4", "0.5", "0.7", "1", "1.5", "2"]

#---------------------------
random_seed = 42 #@param {type:"integer"}
torch_geometric.seed_everything(random_seed)

if not os.path.exists('model_weights'):
  os.makedirs('model_weights')
if not os.path.exists(f'model_weights/model_noised_=_{model_noise}.pt'):
  os.system(f'wget https://github.com/AndreiSokolovskii/hackaton_december_2025/raw/refs/heads/main/attention_visual/{model_noise}.tar.xz -P model_weights')
  os.system(f'cd model_weights; tar -xJf {model_noise}.tar.xz')

device = 'cuda'
checkpoint_path = os.path.join('model_weights',  f'model_noised_=_{model_noise}.pt')
checkpoint = torch.load(checkpoint_path, map_location=device)
model = GraphEncoderBlock_fixed()
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

bias_by_res_dict = None
tied_positions_dict = None
bias_AA_dict = None
omit_AA_dict = None
pssm_dict = None
fixed_positions_dict = None
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'
MAX_NODES_PER_BATCH = 5000
bias_AAs_np = np.zeros(len(alphabet))
omit_AAs_list = 'X' + rm_aa
omit_AAs_np = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)
if bias_AA_dict:
            for n, AA in enumerate(alphabet):
                    if AA in list(bias_AA_dict.keys()):
                            bias_AAs_np[n] = bias_AA_dict[AA]
temperatures = [sampling_temp]

from add_utils import *
pdb_dict_list = parse_PDB(file_path)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)
all_chain_list = [item[-1:] for item in list(pdb_dict_list[0]) if item[:9]=='seq_chain'] #['A','B', 'C',...]

designed_chain_list = [str(item) for item in chains.split()]
fixed_chain_list = [letter for letter in all_chain_list if letter not in designed_chain_list]
chain_id_dict = {}
chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_list, fixed_chain_list)

with torch.inference_mode():
        test_sum, test_weights = 0., 0.
        for ix, protein in enumerate(dataset_valid):
            score_list = []
            global_score_list = []
            all_probs_list = []
            all_log_probs_list = []
            S_sample_list = []
            BATCH_COPIES = num_seq_per_target * len(temperatures)
            out_fasta_name = protein['name']
            batch = tied_featurize_pyg(batch=[copy.deepcopy(protein)],
                                   device=device,
                                   chain_dict=chain_id_dict,
                                   fixed_position_dict=fixed_positions_dict,
                                   omit_AA_dict=omit_AA_dict,
                                   tied_positions_dict=tied_positions_dict,
                                   pssm_dict=pssm_dict,
                                   bias_by_res_dict=bias_by_res_dict)
            batch_clones = [copy.deepcopy(batch) for i in range(BATCH_COPIES)]
            S_for_ref = batch.chain_seq_label.clone()
            global_Batch = Batch.from_data_list(batch_clones)
            clone_chunks = chunk_clones_by_node_budget(batch_clones, MAX_NODES_PER_BATCH)
            all_outputs = []


            for chunk in clone_chunks:


                            sub_batch = Batch.from_data_list(chunk).to(device)
                            pos, chain_seq_label, mask, chain_mask_all, residue_idx, chain_encoding_all = sub_batch.pos, sub_batch.chain_seq_label, sub_batch.mask, sub_batch.chain_mask_all, sub_batch.residue_idx, sub_batch.chain_encoding_all


                            pssm_log_odds_mask = (sub_batch.pssm_log_odds > 0.0).float()
                            output_dict = model(pos=pos,
                                                chain_seq_label=chain_seq_label,
                                                mask=mask,
                                                chain_mask_all=chain_mask_all,
                                                residue_idx=residue_idx,
                                                chain_encoding_all=chain_encoding_all, batch=sub_batch.batch,
                                                force_full_mask=True, return_global_attn=True, return_local_attn=True, collect_raw_stats=True)
logits = output_dict['logits']
att2 = output_dict['attn_s']
raw_stats = output_dict['raw_stats']
att = [a[0] for a in att2]
local_att = [a[-1] for a in att2]

num_heads = 4
attn_global = {}
for i, at in enumerate(att):
    q_proj, k_proj = at
    mean_head = []
    head_attn_scores = []
    for head in range(num_heads):
        attn_scores_head =q_proj[0][head] @ k_proj[0][head].T #at[0][head]
        den = (q_proj[0][head] @ k_proj[0][head].sum(axis=0, keepdims=True).T)
        attn_scores_head = attn_scores_head / (den + 1e-12)
        head_attn_scores += [attn_scores_head]
    attn_global[i] = torch.stack(head_attn_scores, dim=0)  # (H, N, N)
la = {}
for i, at in enumerate(local_att):
    edge_weights = at
    la[i] = edge_weights


In [ ]:
#@markdown ##Attention map visualisation from one *gothrough* the model.
from attn_viz import interactive_attention_viewer
interactive_attention_viewer(
    pdb_string  = open(file_path).read(),
    attn_global = attn_global,
    attn_local  = la,
    chain_id    = chains,
    cmap        = "bwr",
)

In [ ]:
#@markdown ##Attention map visualisation from iterative sampling the model.
from model_lib import GraphEncoderBlock_fixed
from torch_geometric.data import Batch
import os
import copy
from_N_to_C_term_order = True #@param {type:"boolean"}
device = 'cuda'
checkpoint_path = os.path.join('model_weights',  f'model_noised_=_{model_noise}.pt')
checkpoint = torch.load(checkpoint_path, map_location=device)
model = GraphEncoderBlock_fixed()
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()


bias_by_res_dict = None
tied_positions_dict = None
bias_AA_dict = None
omit_AA_dict = None
pssm_dict = None
fixed_positions_dict = None
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'
MAX_NODES_PER_BATCH = 5000
bias_AAs_np = np.zeros(len(alphabet))
omit_AAs_list = 'X' + rm_aa
omit_AAs_np = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)
if bias_AA_dict:
            for n, AA in enumerate(alphabet):
                    if AA in list(bias_AA_dict.keys()):
                            bias_AAs_np[n] = bias_AA_dict[AA]
temperatures = [sampling_temp]
with torch.inference_mode():
        test_sum, test_weights = 0., 0.
        for ix, protein in enumerate(dataset_valid):
            score_list = []
            global_score_list = []
            all_probs_list = []
            all_log_probs_list = []
            S_sample_list = []
            BATCH_COPIES = num_seq_per_target * len(temperatures)
            out_fasta_name = protein['name']
            batch = tied_featurize_pyg(batch=[copy.deepcopy(protein)],
                                   device=device,
                                   chain_dict=chain_id_dict,
                                   fixed_position_dict=fixed_positions_dict,
                                   omit_AA_dict=omit_AA_dict,
                                   tied_positions_dict=tied_positions_dict,
                                   pssm_dict=pssm_dict,
                                   bias_by_res_dict=bias_by_res_dict)
            batch_clones = [copy.deepcopy(batch) for i in range(BATCH_COPIES)]
            S_for_ref = batch.chain_seq_label.clone()
            global_Batch = Batch.from_data_list(batch_clones)
            clone_chunks = chunk_clones_by_node_budget(batch_clones, MAX_NODES_PER_BATCH)
            all_outputs = []
            for chunk in clone_chunks:
                            sub_batch = Batch.from_data_list(chunk).to(device)
                            pos, chain_seq_label, mask, chain_mask_all, residue_idx, chain_encoding_all = sub_batch.pos, sub_batch.chain_seq_label, sub_batch.mask, sub_batch.chain_mask_all, sub_batch.residue_idx, sub_batch.chain_encoding_all
                            pssm_log_odds_mask = (sub_batch.pssm_log_odds > 0.0).float()
                            output_dict = model.sample(pos=pos,
                                                    chain_seq_label=chain_seq_label,
                                                    mask=mask,
                                                    chain_mask_all=chain_mask_all,
                                                    residue_idx=residue_idx,
                                                    chain_encoding_all=chain_encoding_all, batch=sub_batch.batch,
                                                    temperature=temperatures,
                                                    chain_M_pos=sub_batch.chain_M_pos,
                                                    omit_AAs_np=omit_AAs_np,
                                                    bias_AAs_np=bias_AAs_np,
                                                    omit_AA_mask=sub_batch.omit_AA_mask,
                                                    bias_by_res=sub_batch.bias_by_res,
                                                    pssm_coef=sub_batch.pssm_coef,
                                                    pssm_bias=sub_batch.pssm_bias,
                                                    pssm_bias_flag=bool(0),
                                                    pssm_log_odds_flag=bool(0),
                                                    pssm_log_odds_mask=pssm_log_odds_mask,
                                                    pssm_multi=0.0, deterministic=from_N_to_C_term_order, return_local_attn=True, return_global_attn=True,
                                                    collect_raw_stats=True, progress_bar=progress_bar)

heat_map_list_local = output_dict['heat_map_list_local']
heat_map_list_global = output_dict['heat_map_list_global']
heat_map_list_comb = output_dict['heat_map_list_comb']
final_local_attn = output_dict['final_local_attn']
final_global_attn = output_dict['final_global_attn']
raw_stats = output_dict['raw_stats']
still_unpridicted = output_dict['still_unpridicted']
attn_global_steps = output_dict['attn_global_steps']
attn_local_steps = output_dict['attn_local_steps']

In [ ]:
from attn_viz import interactive_step_scrubber
interactive_step_scrubber(pdb_string=open(file_path).read(), attn_global_steps=attn_global_steps, attn_local_steps=attn_local_steps, cmap='bwr')


In [ ]:
#@title Attention map though iteration {run: "auto"}
from attn_viz import get_animation
from IPython.display import HTML
atten_type = 'local' #@param ["local", "global", "comb"]
color = "bwr_r" #@param ["bwr_r", "bwr"]
heat_map_lists = {'local': heat_map_list_local,
                  'global': heat_map_list_global,
                  'comb': heat_map_list_comb}
anim = get_animation(heat_map_lists[atten_type])
HTML(anim.to_jshtml())